In [ ]:
# Sparse High-Order n-gram Language Model with Kneser-Ney Smoothing from Scratch

from collections import defaultdict, Counter
import math

# ---------------------------------------------------
# Sample Training Corpus
# ---------------------------------------------------
corpus = [
    "artificial intelligence is transforming healthcare",
    "artificial intelligence improves education",
    "machine learning is part of artificial intelligence",
    "healthcare uses artificial intelligence",
    "education uses machine learning",
    "artificial intelligence and machine learning are related"
]

# ---------------------------------------------------
# Preprocessing
# ---------------------------------------------------
def preprocess(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]

processed = [preprocess(sent) for sent in corpus]

# ---------------------------------------------------
# Build n-gram Counts
# ---------------------------------------------------
N = 3   # Trigram Model

ngram_counts = defaultdict(Counter)

for sentence in processed:
    for n in range(1, N + 1):
        for i in range(len(sentence) - n + 1):
            context = tuple(sentence[i:i+n-1])
            word = sentence[i+n-1]
            ngram_counts[context][word] += 1

# ---------------------------------------------------
# Vocabulary
# ---------------------------------------------------
vocab = set()

for sentence in processed:
    vocab.update(sentence)

vocab = sorted(vocab)

print("Vocabulary:")
print(vocab)

# ---------------------------------------------------
# Continuation Counts
# ---------------------------------------------------
continuation = defaultdict(set)

for context in ngram_counts:
    for word in ngram_counts[context]:
        continuation[word].add(context)

continuation_count = {
    word: len(continuation[word])
    for word in continuation
}

# ---------------------------------------------------
# Total Continuations
# ---------------------------------------------------
total_continuations = sum(continuation_count.values())

# ---------------------------------------------------
# Kneser-Ney Probability
# ---------------------------------------------------
discount = 0.75

def kneser_ney(context, word):

    context = tuple(context)

    if context not in ngram_counts:
        return continuation_count.get(word,1) / total_continuations

    count_context = sum(ngram_counts[context].values())
    count_word = ngram_counts[context][word]

    unique_follow = len(ngram_counts[context])

    lambda_weight = (discount * unique_follow) / count_context

    lower_order = continuation_count.get(word,1) / total_continuations

    probability = max(count_word-discount,0)/count_context \
                  + lambda_weight*lower_order

    return probability

# ---------------------------------------------------
# Test Sentences
# ---------------------------------------------------
tests = [
    ("artificial","intelligence"),
    ("machine","learning"),
    ("healthcare","uses"),
    ("education","uses"),
    ("artificial","learning")
]

print("\nBigram Probabilities (Kneser-Ney)\n")

for c,w in tests:
    p = kneser_ney([c],w)
    print(f"P({w}|{c}) = {p:.4f}")

# ---------------------------------------------------
# Trigram Example
# ---------------------------------------------------
print("\nTrigram Probability Example\n")

context = ["artificial","intelligence"]
word = "improves"

p = kneser_ney(context,word)

print(f"P({word}|{' '.join(context)}) = {p:.4f}")

# ---------------------------------------------------
# Sentence Probability
# ---------------------------------------------------
def sentence_probability(sentence):

    words = preprocess(sentence)

    probability = 1

    for i in range(2,len(words)):
        context = words[i-2:i]
        word = words[i]

        probability *= kneser_ney(context,word)

    return probability

sentence = "artificial intelligence improves education"

prob = sentence_probability(sentence)

print("\nSentence:",sentence)
print("Sentence Probability:",prob)

# ---------------------------------------------------
# Perplexity
# ---------------------------------------------------
def perplexity(sentence):

    words = preprocess(sentence)

    log_prob = 0

    N = len(words)-2

    for i in range(2,len(words)):
        context = words[i-2:i]
        word = words[i]

        p = kneser_ney(context,word)

        log_prob += math.log2(p)

    return pow(2,-log_prob/N)

pp = perplexity(sentence)

print("Perplexity:",round(pp,3))